In [ ]:
#@title 🎧 Download Narration Audio & Play Introduction: Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_01_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
#@title 🎧 Download Narration Audio & Play Introduction
import os as _os
if not _os.path.exists("/content/narration"):
    !pip install -q gdown
    import gdown
    gdown.download(id="18t7FWmzH6dWbgEkIn2Uz0Z02BPqoU50z", output="/content/narration.zip", quiet=False)
    !unzip -q /content/narration.zip -d /content/narration
    !rm /content/narration.zip
    print(f"Loaded {len(_os.listdir('/content/narration'))} narration segments")
else:
    print("Narration audio already loaded.")

from IPython.display import Audio, display
display(Audio("/content/narration/01_00_setup.mp3"))


In [ ]:
# Setup: Run this cell first!
# Check GPU availability and install dependencies

import torch
import sys
import time

# Check GPU
if torch.cuda.is_available():
    device = torch.device('cuda')
    print(f"\u2705 GPU available: {torch.cuda.get_device_name(0)}")
    print(f"   Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    device = torch.device('cpu')
    print("\u26a0\ufe0f No GPU detected. This notebook runs fine on CPU.")
    print("   Go to Runtime \u2192 Change runtime type \u2192 GPU (optional)")

print(f"\n\U0001f4e6 Python {sys.version.split()[0]}")
print(f"\U0001f525 PyTorch {torch.__version__}")

# Set random seeds for reproducibility
import random
import numpy as np

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print(f"\U0001f3b2 Random seed set to {SEED}")

import matplotlib.pyplot as plt
import math

# Clean plotting style
plt.rcParams.update({
    'figure.facecolor': 'white',
    'axes.facecolor': '#fafafa',
    'axes.grid': True,
    'grid.alpha': 0.3,
    'font.size': 11,
    'figure.dpi': 100,
})

%matplotlib inline

# \u26a1 From Softmax to Linear Attention

*Part 1 of the Vizuara series on Building Qwen3.5 from Scratch*
*Estimated time: ~90 minutes*

Standard self-attention computes an $N \times N$ matrix for every layer \u2014 that is $O(N^2)$ in both compute and memory. For a 100,000-token document, a single attention layer needs 20 GB just for that matrix. Can we do better?

In this notebook, we will:
1. **Implement standard softmax attention** and see exactly where the $O(N^2)$ bottleneck lives
2. **Derive linear attention from first principles** \u2014 replace softmax with a kernel, then flip the multiplication order to get $O(N)$
3. **Build the recurrent view** \u2014 show that linear attention is secretly an RNN with state $S_t = S_{t-1} + \tilde{k}_t v_t^T$
4. **Benchmark and visualize** the speed difference with real timing experiments
5. **See the limitation** of purely additive memory \u2014 motivating the delta rule in Notebook 2

In [ ]:
#@title 🎧 Listen: Ai Assistant
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_02_ai_assistant.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


# \U0001f916 AI Teaching Assistant

Need help with this notebook? Open the **AI Teaching Assistant** \u2014 it has already read this entire notebook and can help with concepts, code, and exercises.

**[\U0001f449 Open AI Teaching Assistant](https://pods.vizuara.ai/courses/build-llm/practice/7/assistant)**

*Tip: Open it in a separate tab and work through this notebook side-by-side.*

In [ ]:
#@title 🎧 Listen: Attention Bottleneck Concept
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_03_attention_bottleneck_concept.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


## 1. \U0001f4a5 The Attention Bottleneck

### How Standard Attention Works

In a standard Transformer, every token in a sequence looks at every other token. If you have a sequence of $N$ tokens, the model computes an $N \times N$ attention matrix. The formula:

$$\text{Attention}(Q, K, V) = \text{softmax}\!\left(\frac{Q K^T}{\sqrt{d_k}}\right) V$$

Here $Q$, $K$, $V$ are matrices of shape $N \times d_k$ (queries, keys, values). The product $QK^T$ is $N \times N$ \u2014 and **this** is where the problem lives.

### The Numerical Reality

Suppose you are processing a document with $N = 100{,}000$ tokens (roughly a 300-page book):

| Quantity | Value |
|----------|-------|
| Attention matrix entries | $100{,}000^2 = 10^{10}$ |
| Memory (float16) | $10^{10} \times 2$ bytes $= 20$ GB |
| Per-layer, per-head | 20 GB |
| 32 layers $\times$ 32 heads | $> 20$ TB |

Even with FlashAttention (which reduces *memory*), the fundamental *computation* is still $O(N^2)$. Double the context window, quadruple the compute.

Let us see this concretely.

In [ ]:
#@title 🎧 Code Walkthrough: Bottleneck Code
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_04_bottleneck_code.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
# The O(N^2) bottleneck in action
print("=" * 60)
print("  THE ATTENTION BOTTLENECK")
print("=" * 60)

# Show how the attention matrix size grows
seq_lengths = [100, 1_000, 10_000, 100_000, 1_000_000]

print(f"\n{'Sequence Length N':>20} {'Attention Matrix':>20} {'Memory (float16)':>18}")
print("-" * 62)

for N in seq_lengths:
    entries = N * N
    memory_bytes = entries * 2  # float16 = 2 bytes
    if memory_bytes < 1e6:
        mem_str = f"{memory_bytes / 1e3:.0f} KB"
    elif memory_bytes < 1e9:
        mem_str = f"{memory_bytes / 1e6:.0f} MB"
    elif memory_bytes < 1e12:
        mem_str = f"{memory_bytes / 1e9:.1f} GB"
    else:
        mem_str = f"{memory_bytes / 1e12:.1f} TB"

    print(f"{N:>20,} {f'{N}x{N}':>20} {mem_str:>18}")

print(f"\n\u274c N=100,000 needs 20 GB just for ONE attention matrix!")
print(f"   A 32-layer, 32-head model would need >20 TB.")
print(f"   This is clearly impractical.")

In [ ]:
#@title 🎧 What to Look For: Bottleneck Viz
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_05_bottleneck_viz.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
# Visualize the quadratic scaling
N_range = np.linspace(1, 200_000, 500)
d_k = 128

# FLOPs for the QK^T multiplication: N * N * d_k
flops_standard = N_range ** 2 * d_k
flops_linear = N_range * d_k * d_k  # We will explain this soon

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Linear scale
axes[0].plot(N_range / 1000, flops_standard / 1e12, color='#e53935', linewidth=2.5,
             label=r'Standard: $O(N^2 d_k)$')
axes[0].plot(N_range / 1000, flops_linear / 1e12, color='#1565c0', linewidth=2.5,
             label=r'Linear: $O(N d_k d_v)$')
axes[0].set_xlabel('Sequence Length N (thousands)', fontsize=12)
axes[0].set_ylabel('FLOPs (trillions)', fontsize=12)
axes[0].set_title('Compute Cost: Standard vs Linear Attention', fontsize=14,
                  fontweight='bold')
axes[0].legend(fontsize=11)

# Log-log scale
axes[1].loglog(N_range, flops_standard, color='#e53935', linewidth=2.5,
               label=r'Standard: $O(N^2 d_k)$')
axes[1].loglog(N_range, flops_linear, color='#1565c0', linewidth=2.5,
               label=r'Linear: $O(N d_k d_v)$')
axes[1].set_xlabel('Sequence Length N', fontsize=12)
axes[1].set_ylabel('FLOPs', fontsize=12)
axes[1].set_title('Log-Log Scale (slopes reveal exponents)', fontsize=14,
                  fontweight='bold')
axes[1].legend(fontsize=11)

plt.suptitle(f'Attention Scaling (d_k = d_v = {d_k})', fontsize=15,
             fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print("Left: On a linear scale, standard attention's cost explodes.")
print("Right: On a log-log scale, the slope of 2 (standard) vs 1 (linear) is clear.")
print(f"\nAt N=100,000 with d_k={d_k}:")
print(f"  Standard: {100_000**2 * d_k:.2e} FLOPs")
print(f"  Linear:   {100_000 * d_k * d_k:.2e} FLOPs")
print(f"  Ratio:    {100_000**2 * d_k / (100_000 * d_k * d_k):.0f}x reduction")

In [ ]:
#@title 🎧 Listen: Key Takeaway Bottleneck
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_06_key_takeaway_bottleneck.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


### Key Takeaway

The $QK^T$ product creates an $N \times N$ matrix \u2014 this is the bottleneck. Everything downstream (multiplying by $V$, etc.) is fine. If we could somehow **avoid** creating this $N \times N$ matrix, we would break the quadratic barrier.

That is exactly what linear attention does.

---


In [ ]:
#@title 🎧 Narration: Softmax Intro Todo1
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_07_softmax_intro_todo1.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


## 2. \U0001f525 Standard Softmax Attention

Before we can improve attention, we need to understand it fully by implementing it ourselves.

### The Full Formula

Given queries $Q$, keys $K$, values $V$ \u2014 all of shape $(N, d_k)$:

1. Compute the raw scores: $\text{scores} = \frac{Q K^T}{\sqrt{d_k}}$ \u2014 shape $(N, N)$
2. Apply softmax **row-wise**: $\text{weights} = \text{softmax}(\text{scores})$ \u2014 shape $(N, N)$
3. Compute the output: $\text{output} = \text{weights} \cdot V$ \u2014 shape $(N, d_k)$

The softmax ensures each row sums to 1 (each token\u2019s attention weights form a probability distribution over all tokens).

### \U0001f527 TODO #1: Implement Standard Softmax Attention

Implement `softmax_attention(Q, K, V)` and visualize the attention matrix.

In [ ]:
#@title 🎧 Before You Start: Todo1 Softmax
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_08_todo1_softmax.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
# ============ TODO ============
# Implement standard softmax attention.
#
# Given:
#   Q: query matrix, shape (N, d_k)
#   K: key matrix, shape (N, d_k)
#   V: value matrix, shape (N, d_v)
#
# Return:
#   output: shape (N, d_v)
#   weights: shape (N, N) -- the attention weight matrix (for visualization)
#
# Steps:
#   1. Compute scores = Q @ K^T / sqrt(d_k)
#   2. Apply softmax row-wise: weights = softmax(scores, dim=-1)
#   3. Compute output = weights @ V
#
# Use torch.softmax(tensor, dim=-1) for the softmax.
# ============ TODO ============

def softmax_attention(Q, K, V):
    '''Compute standard scaled dot-product attention.

    Args:
        Q: Queries, shape (N, d_k)
        K: Keys, shape (N, d_k)
        V: Values, shape (N, d_v)

    Returns:
        output: shape (N, d_v)
        weights: attention matrix, shape (N, N)
    '''
    N, d_k = Q.shape

    # Step 1: Compute raw attention scores
    scores = ???  # YOUR CODE HERE: Q @ K^T / sqrt(d_k)

    # Step 2: Apply softmax row-wise to get attention weights
    weights = ???  # YOUR CODE HERE: softmax over last dimension

    # Step 3: Compute weighted sum of values
    output = ???  # YOUR CODE HERE: weights @ V

    return output, weights

In [ ]:
#@title 🎧 Narration: Todo1 Verification
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_09_todo1_verification.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
# ============ VERIFICATION ============
# Test with a small example where we can verify by hand

torch.manual_seed(42)

# Small example: 4 tokens, dimension 3
N, d_k = 4, 3
Q = torch.randn(N, d_k)
K = torch.randn(N, d_k)
V = torch.randn(N, d_k)

try:
    output, weights = softmax_attention(Q, K, V)

    # Check shapes
    assert output.shape == (N, d_k), f"Output shape should be ({N}, {d_k}), got {output.shape}"
    assert weights.shape == (N, N), f"Weights shape should be ({N}, {N}), got {weights.shape}"

    # Check that weights are valid probabilities (rows sum to 1)
    row_sums = weights.sum(dim=-1)
    assert torch.allclose(row_sums, torch.ones(N), atol=1e-5), \
        f"Attention weights should sum to 1 per row, got {row_sums}"

    # Check all weights are non-negative
    assert (weights >= 0).all(), "Attention weights should be non-negative"

    # Compare with PyTorch reference
    import torch.nn.functional as F
    scores_ref = Q @ K.T / math.sqrt(d_k)
    weights_ref = F.softmax(scores_ref, dim=-1)
    output_ref = weights_ref @ V

    assert torch.allclose(output, output_ref, atol=1e-5), "Output does not match reference!"
    assert torch.allclose(weights, weights_ref, atol=1e-5), "Weights do not match reference!"

    print("\u2705 Shape check passed: output is (N, d_k), weights is (N, N)")
    print("\u2705 Probability check: all rows sum to 1, all values >= 0")
    print("\u2705 Reference check: matches PyTorch implementation")
    print(f"\nAttention weight matrix (4x4):\n{weights}")
    print(f"\nRow sums: {row_sums.tolist()}")
    print("\n\u2705 All checks passed!")

except NameError:
    print("\u274c Replace the ??? placeholders with your code.")
except Exception as e:
    print(f"\u274c Error: {e}")

In [ ]:
#@title 🎧 What to Look For: Todo1 Viz
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_10_todo1_viz.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
# Visualize the attention matrix
# (Run this after completing TODO #1)

try:
    torch.manual_seed(42)
    N_viz, d_k_viz = 32, 16
    Q_viz = torch.randn(N_viz, d_k_viz)
    K_viz = torch.randn(N_viz, d_k_viz)
    V_viz = torch.randn(N_viz, d_k_viz)

    _, weights_viz = softmax_attention(Q_viz, K_viz, V_viz)

    fig, ax = plt.subplots(figsize=(8, 7))
    im = ax.imshow(weights_viz.numpy(), cmap='Blues', aspect='auto')
    ax.set_xlabel('Key Position (j)', fontsize=12)
    ax.set_ylabel('Query Position (i)', fontsize=12)
    ax.set_title(f'Attention Weight Matrix (N={N_viz})\nThis is the O(N\u00b2) bottleneck',
                 fontsize=14, fontweight='bold')
    plt.colorbar(im, ax=ax, label='Attention Weight')
    plt.tight_layout()
    plt.show()

    print(f"Matrix size: {N_viz} x {N_viz} = {N_viz**2} entries")
    print(f"Each row sums to 1 (softmax normalization)")
    print(f"\nThis {N_viz}x{N_viz} matrix is tiny. Imagine N=100,000...")
    print(f"That would be {100_000**2:,} entries = 20 GB in float16!")

except NameError:
    print("Complete TODO #1 first, then run this cell.")

---


In [ ]:
#@title 🎧 Narration: Stop Think Softmax
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_11_stop_think_softmax.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


### \u270b Stop and Think
Before looking at the solution:
1. In step 1, why do we divide by $\sqrt{d_k}$? What would happen without this scaling?
2. Why must we apply softmax *row-wise* (dim=-1) rather than over the entire matrix?
3. The attention matrix is $N \times N$. What is the memory cost of storing it? How does this compare to storing $Q$, $K$, and $V$?

*Take a minute. Then scroll down.*

---

In [ ]:
#@title 🎧 Listen: Softmax Solution Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_12_softmax_solution_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


### Solution

In [ ]:
#@title 🎧 Code Walkthrough: Softmax Solution Code
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_13_softmax_solution_code.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
def softmax_attention(Q, K, V):
    '''Compute standard scaled dot-product attention.

    Attention(Q, K, V) = softmax(Q K^T / sqrt(d_k)) V
    '''
    N, d_k = Q.shape

    # Step 1: Compute raw attention scores -- the N x N matrix
    scores = Q @ K.T / math.sqrt(d_k)  # shape: (N, N)

    # Step 2: Apply softmax row-wise
    weights = torch.softmax(scores, dim=-1)  # shape: (N, N)

    # Step 3: Weighted combination of values
    output = weights @ V  # shape: (N, d_v)

    return output, weights


# Verify
torch.manual_seed(42)
N, d_k = 4, 3
Q = torch.randn(N, d_k)
K = torch.randn(N, d_k)
V = torch.randn(N, d_k)

output, weights = softmax_attention(Q, K, V)

print("Attention weights:")
print(weights)
print(f"\nRow sums: {weights.sum(dim=-1).tolist()}")
print(f"Output shape: {output.shape}")
print(f"\n\u2705 Standard softmax attention implemented correctly.")

---


In [ ]:
#@title 🎧 Listen: Linear Attention Concept
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_14_linear_attention_concept.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


## 3. \u26a1 Linear Attention: Flipping the Multiplication Order

### The Key Insight

In standard attention with softmax, the nonlinearity **couples all tokens together** \u2014 you must compute the full $N \times N$ matrix before you can normalize. But what if we replace softmax with a simpler function?

Let us define a **kernel function** $\phi$ and set:

$$\tilde{Q} = \phi(Q), \quad \tilde{K} = \phi(K)$$

Without softmax, the attention output for query $i$ becomes:

$$\text{output}_i = \frac{\sum_{j=1}^{N} (\tilde{q}_i^T \tilde{k}_j) \cdot v_j}{\sum_{j=1}^{N} \tilde{q}_i^T \tilde{k}_j}$$

Now here is the trick. We can **rearrange the order of multiplication**:

$$\text{output}_i = \frac{\tilde{q}_i^T \left(\sum_{j=1}^{N} \tilde{k}_j v_j^T\right)}{\tilde{q}_i^T \left(\sum_{j=1}^{N} \tilde{k}_j\right)}$$

The term $S = \sum_{j=1}^{N} \tilde{k}_j v_j^T$ is a $d_k \times d_v$ matrix. Crucially:
- It does **not** depend on $i$
- Its size does **not** grow with $N$
- We compute it **once**, then each query just multiplies against it

### The Cost Comparison

| Method | Bottleneck | Cost |
|--------|-----------|------|
| Standard: $(\tilde{Q}\tilde{K}^T)V$ | $N \times N$ matrix | $O(N^2 d_k)$ |
| Linear: $\tilde{Q}(\tilde{K}^T V)$ | $d_k \times d_v$ matrix | $O(N d_k d_v)$ |

With $N = 100{,}000$ and $d_k = d_v = 128$:
- Standard: $100{,}000^2 \times 128 = 1.28 \times 10^{12}$ operations
- Linear: $100{,}000 \times 128 \times 128 = 1.64 \times 10^{9}$ operations
- **Ratio: ~780x reduction!**

### Choice of Kernel $\phi$

The simplest choice is $\phi(x) = \text{elu}(x) + 1$, which ensures all values are positive (needed because we want the "attention weights" to be non-negative). For this notebook we will use this simple kernel, but note that Qwen3.5 uses a different approach (L2-normalized keys and queries without an explicit kernel).

In [ ]:
#@title 🎧 Code Walkthrough: Reordering Demo
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_15_reordering_demo.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
# Demonstrate the reordering trick with a concrete example
print("=" * 60)
print("  THE REORDERING TRICK")
print("=" * 60)

torch.manual_seed(42)
N, d_k = 6, 4

Q = torch.randn(N, d_k)
K = torch.randn(N, d_k)
V = torch.randn(N, d_k)

# Kernel function: elu(x) + 1 (ensures positivity)
def phi(x):
    '''Simple kernel: elu(x) + 1. Ensures all values > 0.'''
    return torch.nn.functional.elu(x) + 1

Q_tilde = phi(Q)  # (N, d_k)
K_tilde = phi(K)  # (N, d_k)

# === Method 1: Standard order -- (Q_tilde @ K_tilde^T) @ V ===
# Step 1: N x N attention matrix
attn_matrix = Q_tilde @ K_tilde.T  # (N, N) -- THIS is the bottleneck
# Step 2: Normalize rows
attn_normalized = attn_matrix / attn_matrix.sum(dim=-1, keepdim=True)
# Step 3: Multiply by V
output_standard = attn_normalized @ V  # (N, d_k)

print(f"\nMethod 1: (Q_tilde @ K_tilde^T) @ V")
print(f"  Intermediate: {N}x{N} attention matrix = {N*N} entries")

# === Method 2: Reordered -- Q_tilde @ (K_tilde^T @ V) ===
# Step 1: d_k x d_v state matrix (does NOT grow with N!)
S = K_tilde.T @ V  # (d_k, d_v)
# Step 2: Each query multiplies against the fixed-size state
numerator = Q_tilde @ S  # (N, d_v)
# Step 3: Normalize
z = Q_tilde @ K_tilde.sum(dim=0, keepdim=True).T  # (N, 1) normalizer
output_linear = numerator / z

print(f"\nMethod 2: Q_tilde @ (K_tilde^T @ V)")
print(f"  Intermediate: {d_k}x{d_k} state matrix = {d_k*d_k} entries")

# Check: both methods produce the same output
print(f"\n--- Comparison ---")
print(f"Max absolute difference: {(output_standard - output_linear).abs().max().item():.2e}")
match = torch.allclose(output_standard, output_linear, atol=1e-5)
print(f"Are they equal? {'Yes!' if match else 'No'}")

print(f"\n\u2728 Same output, but Method 2 never creates the N x N matrix!")
print(f"   For N=100,000 and d=128:")
print(f"   Method 1: {100_000 * 100_000:>15,} entries (N x N)")
print(f"   Method 2: {128 * 128:>15,} entries (d x d)")
print(f"   Ratio:    {100_000 * 100_000 / (128 * 128):>15,.0f}x smaller")

In [ ]:
#@title 🎧 Listen: Why Reorder State Matrix
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_16_why_reorder_state_matrix.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


### Why Can We Reorder?

The key mathematical identity is the **associativity of matrix multiplication**:

$$(AB)C = A(BC)$$

In standard softmax attention, the softmax **breaks** this associativity \u2014 you must compute $QK^T$ first to apply the row-wise nonlinearity. But with a simple elementwise kernel $\phi$, there is no coupling between rows, so we can freely choose the multiplication order.

This is the entire insight behind linear attention. It is not a new architecture \u2014 it is just a different order of multiplying the same matrices.

### The State Matrix $S$

The matrix $S = \tilde{K}^T V$ has a beautiful interpretation. Each column of $S$ is a weighted sum of all value vectors, weighted by the corresponding key dimension. It is a **compressed summary** of the entire key-value sequence, packed into a $d_k \times d_v$ matrix.

For Qwen3.5 with $d_k = d_v = 128$, this state matrix is just $128 \times 128 \times 4 = 64$ KB \u2014 the same size regardless of whether you have seen 100 tokens or 1,000,000 tokens.

---


In [ ]:
#@title 🎧 Narration: Todo2 Linear Attention Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_17_todo2_linear_attention_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


## 4. \U0001f527 TODO #2: Implement Linear Attention

Implement `linear_attention(Q, K, V, phi)` that uses the **reordered** multiplication ($K^T V$ first). Then compare its output and runtime against standard softmax attention.

Key requirements:
- Use the kernel function $\phi$ to transform Q and K
- Compute the state matrix $S = \tilde{K}^T V$ \u2014 shape $(d_k, d_v)$
- The output for each query is $\tilde{q}_i^T S$, normalized by $\tilde{q}_i^T z$ where $z = \sum_j \tilde{k}_j$

In [ ]:
#@title 🎧 Before You Start: Todo2 Linear Attention
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_18_todo2_linear_attention.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
# ============ TODO ============
# Implement linear attention using the reordered multiplication.
#
# Given:
#   Q: query matrix, shape (N, d_k)
#   K: key matrix, shape (N, d_k)
#   V: value matrix, shape (N, d_v)
#   phi: kernel function (callable)
#
# Return:
#   output: shape (N, d_v) -- the attention output
#   S: shape (d_k, d_v) -- the state matrix (for inspection)
#
# Steps:
#   1. Apply kernel: Q_tilde = phi(Q), K_tilde = phi(K)
#   2. Compute state: S = K_tilde^T @ V          -- shape (d_k, d_v)
#   3. Compute numerator: num = Q_tilde @ S       -- shape (N, d_v)
#   4. Compute denominator: z = Q_tilde @ K_tilde.sum(dim=0)  -- shape (N,)
#   5. Normalize: output = num / z.unsqueeze(-1)
#
# IMPORTANT: The state S has shape (d_k, d_v), NOT (N, N).
# ============ TODO ============

def phi(x):
    '''Simple kernel: elu(x) + 1. Ensures all values > 0.'''
    return torch.nn.functional.elu(x) + 1


def linear_attention(Q, K, V, kernel=phi):
    '''Compute linear attention via the reordered multiplication trick.

    Args:
        Q: Queries, shape (N, d_k)
        K: Keys, shape (N, d_k)
        V: Values, shape (N, d_v)
        kernel: Feature map function phi

    Returns:
        output: shape (N, d_v)
        S: state matrix, shape (d_k, d_v)
    '''
    # Step 1: Apply kernel to get non-negative features
    Q_tilde = ???  # YOUR CODE HERE
    K_tilde = ???  # YOUR CODE HERE

    # Step 2: Compute the state matrix S = K_tilde^T @ V
    # This is the KEY step -- shape (d_k, d_v), NOT (N, N)!
    S = ???  # YOUR CODE HERE

    # Step 3: Compute unnormalized output = Q_tilde @ S
    numerator = ???  # YOUR CODE HERE

    # Step 4: Compute normalizer z = Q_tilde @ sum(K_tilde)
    z = ???  # YOUR CODE HERE: should be shape (N,)

    # Step 5: Normalize
    output = ???  # YOUR CODE HERE: numerator / z (broadcast correctly!)

    return output, S

In [ ]:
#@title 🎧 Narration: Todo2 Verification
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_19_todo2_verification.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
# ============ VERIFICATION ============

torch.manual_seed(42)
N, d_k = 8, 4
Q = torch.randn(N, d_k)
K = torch.randn(N, d_k)
V = torch.randn(N, d_k)

try:
    output_lin, S = linear_attention(Q, K, V)

    # Check shapes
    assert output_lin.shape == (N, d_k), f"Output shape should be ({N}, {d_k}), got {output_lin.shape}"
    assert S.shape == (d_k, d_k), f"State shape should be ({d_k}, {d_k}), got {S.shape}"
    print(f"\u2705 Shape check: output is ({N}, {d_k}), state S is ({d_k}, {d_k})")

    # Compare with brute-force (standard order) for correctness
    Q_t = phi(Q)
    K_t = phi(K)
    attn = Q_t @ K_t.T  # (N, N) -- the "slow" way
    attn_norm = attn / attn.sum(dim=-1, keepdim=True)
    output_ref = attn_norm @ V

    diff = (output_lin - output_ref).abs().max().item()
    assert diff < 1e-5, f"Output differs from reference by {diff}"
    print(f"\u2705 Correctness check: matches brute-force (max diff = {diff:.2e})")

    print(f"\nState matrix S ({d_k}x{d_k}):")
    print(S)
    print(f"\nKey insight: S has {d_k*d_k} entries regardless of N.")
    print(f"For N=100,000: standard creates {100_000*100_000:,} entries, linear creates {d_k*d_k}.")

    print(f"\n\u2705 All checks passed!")

except NameError:
    print("\u274c Replace the ??? placeholders with your code.")
except Exception as e:
    print(f"\u274c Error: {e}")

In [ ]:
#@title 🎧 What to Look For: Timing Comparison
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_20_timing_comparison.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
# Timing comparison: softmax vs linear attention
# (Run after completing both TODO #1 and TODO #2)

try:
    seq_lengths = [100, 500, 1000, 2000, 5000]
    d_k_bench = 64
    n_trials = 5

    softmax_times = []
    linear_times = []

    print(f"{'N':>8} {'Softmax (ms)':>14} {'Linear (ms)':>14} {'Speedup':>10}")
    print("-" * 50)

    for N_bench in seq_lengths:
        torch.manual_seed(42)
        Q_b = torch.randn(N_bench, d_k_bench)
        K_b = torch.randn(N_bench, d_k_bench)
        V_b = torch.randn(N_bench, d_k_bench)

        # Time softmax attention
        times_s = []
        for _ in range(n_trials):
            start = time.perf_counter()
            _ = softmax_attention(Q_b, K_b, V_b)
            times_s.append((time.perf_counter() - start) * 1000)
        t_softmax = np.median(times_s)

        # Time linear attention
        times_l = []
        for _ in range(n_trials):
            start = time.perf_counter()
            _ = linear_attention(Q_b, K_b, V_b)
            times_l.append((time.perf_counter() - start) * 1000)
        t_linear = np.median(times_l)

        softmax_times.append(t_softmax)
        linear_times.append(t_linear)

        speedup = t_softmax / t_linear if t_linear > 0 else float('inf')
        print(f"{N_bench:>8} {t_softmax:>14.2f} {t_linear:>14.2f} {speedup:>9.1f}x")

    # Plot
    fig, ax = plt.subplots(figsize=(10, 6))
    ax.plot(seq_lengths, softmax_times, 'o-', color='#e53935', linewidth=2.5,
            markersize=8, label='Softmax Attention')
    ax.plot(seq_lengths, linear_times, 's-', color='#1565c0', linewidth=2.5,
            markersize=8, label='Linear Attention')
    ax.set_xlabel('Sequence Length N', fontsize=12)
    ax.set_ylabel('Time (ms)', fontsize=12)
    ax.set_title(f'Timing: Softmax vs Linear Attention (d={d_k_bench})',
                 fontsize=14, fontweight='bold')
    ax.legend(fontsize=11)
    plt.tight_layout()
    plt.show()

    print(f"\n\u2728 Linear attention scales much better!")
    print(f"   Softmax time grows as O(N\u00b2); linear time grows as O(N).")
    print(f"   The gap widens as N increases -- exactly when it matters most.")

except NameError:
    print("Complete TODO #1 and #2 first.")

---


In [ ]:
#@title 🎧 Narration: Stop Think Linear
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_21_stop_think_linear.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


### \u270b Stop and Think
Before looking at the solution:
1. In your implementation, what is the shape of the state matrix $S$? Does it depend on $N$?
2. Why is the normalization step (dividing by $z$) necessary? What would happen without it?
3. If $d_k = d_v = 128$ and $N = 100{,}000$, how many entries does $S$ have vs the softmax attention matrix?

*Take a minute. Then scroll down.*

---

In [ ]:
#@title 🎧 Listen: Linear Solution Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_22_linear_solution_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


### Solution

In [ ]:
#@title 🎧 Code Walkthrough: Linear Solution Code
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_23_linear_solution_code.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
def phi(x):
    '''Simple kernel: elu(x) + 1. Ensures all values > 0.'''
    return torch.nn.functional.elu(x) + 1


def linear_attention(Q, K, V, kernel=phi):
    '''Compute linear attention via the reordered multiplication trick.

    Instead of (Q_tilde @ K_tilde^T) @ V  -- O(N^2 d_k)
    We compute  Q_tilde @ (K_tilde^T @ V)  -- O(N d_k d_v)
    '''
    # Step 1: Apply kernel
    Q_tilde = kernel(Q)  # (N, d_k)
    K_tilde = kernel(K)  # (N, d_k)

    # Step 2: State matrix S = K_tilde^T @ V -- the key insight!
    S = K_tilde.T @ V  # (d_k, d_v) -- does NOT depend on N!

    # Step 3: Unnormalized output
    numerator = Q_tilde @ S  # (N, d_v)

    # Step 4: Normalizer
    z = Q_tilde @ K_tilde.sum(dim=0)  # (N,)

    # Step 5: Normalize
    output = numerator / z.unsqueeze(-1)  # (N, d_v)

    return output, S


# Verify
torch.manual_seed(42)
N, d_k = 8, 4
Q = torch.randn(N, d_k)
K = torch.randn(N, d_k)
V = torch.randn(N, d_k)

output_lin, S = linear_attention(Q, K, V)

print(f"Output shape: {output_lin.shape}")
print(f"State S shape: {S.shape}")
print(f"State S has {S.numel()} entries (fixed, regardless of N)")
print(f"\n\u2705 Linear attention: O(N d_k d_v) instead of O(N\u00b2 d_k)")

---


In [ ]:
#@title 🎧 Listen: Recurrent View Concept
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_24_recurrent_view_concept.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


## 5. \U0001f501 The Recurrent View: Linear Attention as an RNN

### From Batch to Recurrence

So far we computed $S = \sum_{j=1}^{N} \tilde{k}_j v_j^T$ as a **batch** operation (one big matrix multiply). But we can also build $S$ **incrementally**, one token at a time:

$$S_t = S_{t-1} + \tilde{k}_t v_t^T$$

$$\text{output}_t = \tilde{q}_t^T S_t$$

This is exactly a **recurrent neural network**! At each time step:
1. The state $S_t$ is a $d_k \times d_v$ matrix \u2014 the model\u2019s "memory"
2. Each new token **writes** to the memory: $S_t = S_{t-1} + \tilde{k}_t v_t^T$
3. The output **reads** from the memory: $\text{output}_t = \tilde{q}_t^T S_t$

### Why This Matters for Inference

During inference (generating one token at a time), this recurrent view is incredibly efficient:
- **Standard attention**: Must attend to ALL previous tokens. Cost: $O(N \cdot d_k)$ per token. Must store the full KV cache.
- **Linear attention**: Just update the state and query it. Cost: $O(d_k \cdot d_v)$ per token \u2014 **independent of $N$!**

| Property | Standard Attention | Linear Attention (Recurrent) |
|----------|-------------------|------------------------------|
| Cost per token | $O(N \cdot d_k)$ | $O(d_k \cdot d_v)$ |
| Memory | $O(N \cdot d_k)$ (KV cache) | $O(d_k \cdot d_v)$ (state matrix) |
| Depends on $N$? | Yes \u2014 grows linearly | No \u2014 constant! |

For Qwen3.5 with $d_k = d_v = 128$ and 64 heads, the state per layer is:
$$64 \times 128 \times 128 \times 2 = 4 \text{ MB}$$
Compare that to the KV cache for 100,000 tokens: $2 \times 100{,}000 \times 128 \times 64 \times 2 \approx 3.3$ GB.

In [ ]:
#@title 🎧 Code Walkthrough: Batch Vs Recurrent Demo
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_25_batch_vs_recurrent_demo.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
# Show that batch and recurrent views produce the same result
print("=" * 60)
print("  BATCH vs RECURRENT: Same Output, Different Process")
print("=" * 60)

torch.manual_seed(42)
N, d_k = 10, 4

Q = torch.randn(N, d_k)
K = torch.randn(N, d_k)
V = torch.randn(N, d_k)

Q_tilde = phi(Q)
K_tilde = phi(K)

# === Batch view: compute S all at once ===
S_batch = K_tilde.T @ V  # (d_k, d_v)
output_batch = Q_tilde @ S_batch  # unnormalized, for comparison

# === Recurrent view: build S token by token ===
S_recurrent = torch.zeros(d_k, d_k)
output_recurrent = torch.zeros(N, d_k)

print(f"\nBuilding state S one token at a time:")
for t in range(N):
    # Write: add new key-value pair to memory
    S_recurrent = S_recurrent + torch.outer(K_tilde[t], V[t])

    # Read: query the current memory
    output_recurrent[t] = Q_tilde[t] @ S_recurrent

    if t < 5 or t == N - 1:  # Print first 5 and last
        norm = torch.norm(S_recurrent).item()
        print(f"  t={t}: S norm = {norm:.4f}, new entry = k_{t} v_{t}^T")

# Compare final states (batch is the sum of all, recurrent should match)
diff_state = (S_batch - S_recurrent).abs().max().item()
print(f"\n--- State Comparison ---")
print(f"Max state difference: {diff_state:.2e}")
print(f"Match: {'\u2705 Exact!' if diff_state < 1e-5 else '\u274c Mismatch'}")

# Note: the output comparison is trickier because recurrent output at time t
# uses S_t (sum up to t), while batch output uses S_N (sum of all).
# For the final token, they should match:
diff_last = (output_batch[-1] - output_recurrent[-1]).abs().max().item()
print(f"Last token output diff: {diff_last:.2e}")

print(f"\n\u2728 Key insight: batch and recurrent build the SAME state matrix.")
print(f"   Batch = one big multiply (good for training)")
print(f"   Recurrent = token-by-token updates (good for inference)")

---


In [ ]:
#@title 🎧 Narration: Todo3 Recurrent Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_26_todo3_recurrent_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


## 6. \U0001f527 TODO #3: Implement the Recurrent View

Implement `linear_attention_recurrent(queries, keys, values, kernel)` that processes tokens **one at a time**, maintaining the state matrix $S_t$.

Requirements:
- Process tokens sequentially: at each step $t$, update $S_t = S_{t-1} + \tilde{k}_t v_t^T$
- Compute the output for token $t$: $\text{output}_t = \tilde{q}_t^T S_t$
- Return the full sequence of outputs AND a list of state snapshots (for visualization)
- Show that this produces the **same** output as the batch linear attention

This exercise cements the connection between linear attention and RNNs.

In [ ]:
#@title 🎧 Before You Start: Todo3 Recurrent
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_27_todo3_recurrent.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
# ============ TODO ============
# Implement the recurrent view of linear attention.
#
# Given:
#   queries: shape (N, d_k)
#   keys: shape (N, d_k)
#   values: shape (N, d_v)
#   kernel: feature map function phi
#
# Return:
#   outputs: shape (N, d_v) -- output at each time step
#   states: list of N tensors, each shape (d_k, d_v) -- state at each step
#
# At each time step t:
#   1. Apply kernel: q_t = kernel(queries[t]), k_t = kernel(keys[t])
#   2. Update state: S = S + outer(k_t, values[t])
#   3. Compute output: output_t = S^T @ q_t (unnormalized)
#   4. Normalize: output_t = output_t / (k_sum^T @ q_t)
#      where k_sum = running sum of all k_tilde so far
#   5. Save state snapshot
#
# Hint: Keep a running sum of k_tilde vectors for normalization.
# ============ TODO ============

def linear_attention_recurrent(queries, keys, values, kernel=phi):
    '''Compute linear attention token-by-token (recurrent view).

    Args:
        queries: shape (N, d_k)
        keys: shape (N, d_k)
        values: shape (N, d_v)
        kernel: feature map phi

    Returns:
        outputs: shape (N, d_v) -- output at each time step
        states: list of N tensors, each shape (d_k, d_v)
    '''
    N, d_k = queries.shape
    d_v = values.shape[1]

    # Initialize state and accumulators
    S = torch.zeros(d_k, d_v)       # The memory state
    k_sum = torch.zeros(d_k)        # Running sum of keys (for normalization)
    outputs = torch.zeros(N, d_v)
    states = []

    for t in range(N):
        # Apply kernel to this token's query and key
        q_t = ???  # YOUR CODE HERE: kernel(queries[t])
        k_t = ???  # YOUR CODE HERE: kernel(keys[t])
        v_t = values[t]

        # Update the running key sum (for normalization)
        k_sum = ???  # YOUR CODE HERE: accumulate k_t

        # Update the state matrix: S = S + k_t v_t^T
        S = ???  # YOUR CODE HERE

        # Compute unnormalized output: S^T @ q_t
        raw_output = ???  # YOUR CODE HERE

        # Normalize by the sum of key similarities
        z_t = ???  # YOUR CODE HERE: dot product of k_sum and q_t

        # Store normalized output
        outputs[t] = ???  # YOUR CODE HERE: raw_output / z_t

        # Save a snapshot of the current state
        states.append(S.clone())

    return outputs, states

In [ ]:
#@title 🎧 Narration: Todo3 Verification
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_28_todo3_verification.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
# ============ VERIFICATION ============
# Check that recurrent view matches batch view

torch.manual_seed(42)
N, d_k = 20, 8
Q = torch.randn(N, d_k)
K = torch.randn(N, d_k)
V = torch.randn(N, d_k)

try:
    # Batch linear attention (reference)
    output_batch, S_batch = linear_attention(Q, K, V)

    # Recurrent linear attention (your implementation)
    output_recurrent, states = linear_attention_recurrent(Q, K, V)

    # The recurrent output at time t uses S_t (sum up to t).
    # The batch output uses S_N (sum of ALL tokens).
    # They only match for the LAST token (when S_t = S_N).
    # But we can compare with a reference recurrent implementation.

    # Reference: manual recurrent
    Q_t = phi(Q)
    K_t = phi(K)
    S_ref = torch.zeros(d_k, d_k)
    k_sum_ref = torch.zeros(d_k)
    output_ref = torch.zeros(N, d_k)
    for t in range(N):
        k_sum_ref = k_sum_ref + K_t[t]
        S_ref = S_ref + torch.outer(K_t[t], V[t])
        raw = S_ref.T @ Q_t[t]
        z = k_sum_ref @ Q_t[t]
        output_ref[t] = raw / z

    diff = (output_ref - output_recurrent).abs().max().item()
    print(f"--- Output Comparison ---")
    print(f"Recurrent output shape: {output_recurrent.shape}")
    print(f"Max diff vs reference:  {diff:.2e}")
    assert diff < 1e-4, f"Outputs differ by {diff}!"
    print(f"\u2705 Outputs match!")

    # Check state shapes
    assert len(states) == N, f"Expected {N} states, got {len(states)}"
    assert states[-1].shape == (d_k, d_k), f"State shape should be ({d_k},{d_k})"
    print(f"\n--- State Properties ---")
    print(f"Number of state snapshots: {len(states)}")
    print(f"Each state shape: {states[0].shape}")

    # Check that final recurrent state matches batch state
    diff_state = (states[-1] - S_batch).abs().max().item()
    print(f"Final state vs batch S: max diff = {diff_state:.2e}")
    assert diff_state < 1e-5, f"Final states differ!"
    print(f"\u2705 Final state matches batch computation!")

    # Show state growth
    print(f"\n--- State Growth Over Time ---")
    for i in [0, 4, 9, 14, N-1]:
        norm = torch.norm(states[i]).item()
        print(f"  t={i:>2}: ||S_t|| = {norm:.4f}")

    print(f"\n\u2705 All checks passed! The recurrent view matches batch.")

except NameError:
    print("\u274c Replace the ??? placeholders with your code.")
except Exception as e:
    print(f"\u274c Error: {e}")

In [ ]:
#@title 🎧 What to Look For: State Evolution Viz
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_29_state_evolution_viz.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
# Visualize the state matrix evolution
# (Run after completing TODO #3)

try:
    # Use a smaller dimension for clear visualization
    torch.manual_seed(42)
    N_viz, d_viz = 50, 8
    Q_v = torch.randn(N_viz, d_viz)
    K_v = torch.randn(N_viz, d_viz)
    V_v = torch.randn(N_viz, d_viz)

    _, states_viz = linear_attention_recurrent(Q_v, K_v, V_v)

    # Show state at different time steps
    time_steps = [0, 4, 12, 24, 49]
    fig, axes = plt.subplots(1, len(time_steps), figsize=(20, 4))

    # Find global color range
    vmax = max(abs(states_viz[t]).max().item() for t in time_steps)

    for idx, t in enumerate(time_steps):
        im = axes[idx].imshow(states_viz[t].numpy(), cmap='RdBu_r',
                              vmin=-vmax, vmax=vmax, aspect='auto')
        axes[idx].set_title(f't = {t}\n||S|| = {torch.norm(states_viz[t]).item():.2f}',
                            fontsize=12, fontweight='bold')
        axes[idx].set_xlabel('Value dim')
        if idx == 0:
            axes[idx].set_ylabel('Key dim')

    plt.colorbar(im, ax=axes, shrink=0.8, label='Entry value')
    plt.suptitle('State Matrix S_t Evolution Over Time', fontsize=15,
                 fontweight='bold', y=1.05)
    plt.tight_layout()
    plt.show()

    # Plot state norm over time
    norms = [torch.norm(s).item() for s in states_viz]

    fig, ax = plt.subplots(figsize=(10, 4))
    ax.plot(range(N_viz), norms, color='#1565c0', linewidth=2.5)
    ax.set_xlabel('Time Step t', fontsize=12)
    ax.set_ylabel('||S_t|| (Frobenius norm)', fontsize=12)
    ax.set_title('State Matrix Growth: ||S_t|| Over Time', fontsize=14,
                 fontweight='bold')
    ax.fill_between(range(N_viz), norms, alpha=0.15, color='#1565c0')
    plt.tight_layout()
    plt.show()

    print("The state matrix grows monotonically -- it only accumulates, never forgets.")
    print("Each token adds its k_t v_t^T to the state, and nothing is ever removed.")
    print("\nThis is the fundamental limitation that motivates the delta rule (Notebook 2).")

except NameError:
    print("Complete TODO #3 first.")

---


In [ ]:
#@title 🎧 Narration: Stop Think Recurrent
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_30_stop_think_recurrent.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


### \u270b Stop and Think
Before looking at the solution:
1. During inference, the recurrent view processes one token at a time. What is the cost per token? Does it depend on how many tokens came before?
2. The state $S_t$ only grows (via addition). What does this mean for long sequences? Can the model ever "forget" old information?
3. Compare the state matrix ($d_k \times d_v$) to the KV cache in standard attention ($N \times d_k$ for keys, $N \times d_v$ for values). Which grows with $N$?

*Take a minute. Then scroll down.*

---

In [ ]:
#@title 🎧 Listen: Recurrent Solution Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_31_recurrent_solution_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


### Solution

In [ ]:
#@title 🎧 Code Walkthrough: Recurrent Solution Code
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_32_recurrent_solution_code.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
def linear_attention_recurrent(queries, keys, values, kernel=phi):
    '''Compute linear attention token-by-token (recurrent view).

    At each step t:
        S_t = S_{t-1} + k_tilde_t @ v_t^T    (write to memory)
        output_t = S_t^T @ q_tilde_t           (read from memory)
    '''
    N, d_k = queries.shape
    d_v = values.shape[1]

    # Initialize
    S = torch.zeros(d_k, d_v)
    k_sum = torch.zeros(d_k)
    outputs = torch.zeros(N, d_v)
    states = []

    for t in range(N):
        # Apply kernel
        q_t = kernel(queries[t])  # (d_k,)
        k_t = kernel(keys[t])    # (d_k,)
        v_t = values[t]          # (d_v,)

        # Accumulate key sum for normalization
        k_sum = k_sum + k_t

        # Write to memory: S = S + k_t v_t^T
        S = S + torch.outer(k_t, v_t)

        # Read from memory: output = S^T @ q_t
        raw_output = S.T @ q_t  # (d_v,)

        # Normalize
        z_t = k_sum @ q_t  # scalar

        # Store
        outputs[t] = raw_output / z_t

        # Snapshot
        states.append(S.clone())

    return outputs, states


# Verify: batch vs recurrent
torch.manual_seed(42)
N, d_k = 20, 8
Q = torch.randn(N, d_k)
K = torch.randn(N, d_k)
V = torch.randn(N, d_k)

output_batch, S_batch = linear_attention(Q, K, V)
output_recurrent, states = linear_attention_recurrent(Q, K, V)

# Final states should match
diff = (S_batch - states[-1]).abs().max().item()
print(f"Batch vs Recurrent final state diff: {diff:.2e}")
print(f"\u2705 {'Match!' if diff < 1e-4 else 'Mismatch!'}")
print(f"\nKey point: the recurrent view processes ONE token at a time.")
print(f"Cost per token: O(d_k * d_v) = O({d_k} * {d_k}) = {d_k * d_k} ops")
print(f"This is CONSTANT -- does not depend on sequence length N!")

---


In [ ]:
#@title 🎧 Listen: Scaling Picture Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_33_scaling_picture_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


## 7. \U0001f4ca The Complete Scaling Picture

Let us bring together everything we have learned with a comprehensive visualization comparing standard and linear attention across multiple dimensions.

In [ ]:
#@title 🎧 What to Look For: Comprehensive Viz
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_34_comprehensive_viz.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
# Comprehensive comparison: memory, compute, and inference cost
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

d_k = 128
N_range = np.logspace(2, 6, 100)  # 100 to 1,000,000

# Panel 1: Training FLOPs (per layer)
flops_standard = N_range ** 2 * d_k
flops_linear = N_range * d_k * d_k

axes[0].loglog(N_range, flops_standard, color='#e53935', linewidth=2.5,
               label=r'Standard: $O(N^2 d)$')
axes[0].loglog(N_range, flops_linear, color='#1565c0', linewidth=2.5,
               label=r'Linear: $O(N d^2)$')

# Mark the crossover point
crossover_N = d_k  # N^2 d = N d^2 when N = d
axes[0].axvline(x=crossover_N, color='gray', linestyle=':', alpha=0.7)
axes[0].annotate(f'N = d = {d_k}\n(crossover)', xy=(crossover_N, 1e8),
                fontsize=9, ha='center', color='gray')

axes[0].set_xlabel('Sequence Length N', fontsize=12)
axes[0].set_ylabel('FLOPs per Layer', fontsize=12)
axes[0].set_title('Training Compute', fontsize=13, fontweight='bold')
axes[0].legend(fontsize=10)

# Panel 2: Memory for attention/state
mem_standard = N_range ** 2 * 2  # float16 bytes for N x N matrix
mem_linear = np.full_like(N_range, d_k * d_k * 4)  # float32 state matrix

axes[1].loglog(N_range, mem_standard, color='#e53935', linewidth=2.5,
               label=r'Standard: $O(N^2)$ attention matrix')
axes[1].loglog(N_range, mem_linear, color='#1565c0', linewidth=2.5,
               label=r'Linear: $O(d^2)$ state matrix')

axes[1].set_xlabel('Sequence Length N', fontsize=12)
axes[1].set_ylabel('Memory (bytes)', fontsize=12)
axes[1].set_title('Memory Footprint', fontsize=13, fontweight='bold')
axes[1].legend(fontsize=10)

# Panel 3: Inference cost per token
inference_standard = N_range * d_k  # Must attend to all N previous tokens
inference_linear = np.full_like(N_range, d_k * d_k)  # Just update state + query

axes[2].loglog(N_range, inference_standard, color='#e53935', linewidth=2.5,
               label=r'Standard: $O(N \cdot d)$ per token')
axes[2].loglog(N_range, inference_linear, color='#1565c0', linewidth=2.5,
               label=r'Linear: $O(d^2)$ per token')

axes[2].set_xlabel('Sequence Length N', fontsize=12)
axes[2].set_ylabel('FLOPs per Token', fontsize=12)
axes[2].set_title('Inference Cost', fontsize=13, fontweight='bold')
axes[2].legend(fontsize=10)

plt.suptitle(f'Standard vs Linear Attention: Full Comparison (d = {d_k})',
             fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print("Left:   During training, linear attention wins when N > d (which is almost always).")
print("Middle: The state matrix is fixed-size; the attention matrix grows as N\u00b2.")
print("Right:  During inference, linear attention costs O(1) per token -- constant!")
print(f"\nAt N = 100,000 and d = {d_k}:")
print(f"  Training speedup:  ~{100_000 / d_k:.0f}x")
print(f"  Memory savings:    ~{100_000**2 / (d_k**2):.0f}x")
print(f"  Inference speedup: ~{100_000 / d_k:.0f}x")

---


In [ ]:
#@title 🎧 Listen: Limitation Concept
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_35_limitation_concept.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


## 8. \u26a0\ufe0f The Limitation: A Memory That Never Forgets

We have seen that linear attention is fast and elegant. But there is a catch.

Look at the state update rule:

$$S_t = S_{t-1} + \tilde{k}_t v_t^T$$

This is **purely additive**. Every token writes to the memory, but nothing is ever erased. Over time, the state becomes a superposition of ALL previous tokens \u2014 old and new, relevant and irrelevant.

This is like a whiteboard that you can only write on, never erase. Eventually the board is covered in overlapping notes and you cannot read anything clearly.

Let us see this concretely.

In [ ]:
#@title 🎧 What to Look For: Memory Degradation Viz
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_36_memory_degradation_viz.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
# Demonstrate the memory degradation problem
print("=" * 60)
print("  THE FORGETTING PROBLEM")
print("=" * 60)

# Simple key-value memory experiment
# Store N random key-value pairs, then try to retrieve them
d_k, d_v = 32, 32

def test_retrieval(n_pairs, d_k=32, d_v=32, seed=42):
    '''Store n_pairs and measure retrieval quality.'''
    torch.manual_seed(seed)

    keys = torch.randn(n_pairs, d_k)
    keys = keys / keys.norm(dim=1, keepdim=True)  # Normalize
    values = torch.randn(n_pairs, d_v)

    # Build state via additive updates (linear attention recurrence)
    S = torch.zeros(d_k, d_v)
    for i in range(n_pairs):
        S = S + torch.outer(keys[i], values[i])

    # Try to retrieve each value
    total_mse = 0.0
    for i in range(n_pairs):
        retrieved = S.T @ keys[i]
        mse = ((retrieved - values[i]) ** 2).mean().item()
        total_mse += mse

    return total_mse / n_pairs

# Test with increasing numbers of stored pairs
test_sizes = [5, 10, 20, 50, 100, 200, 500]
mses = [test_retrieval(n) for n in test_sizes]

print(f"\n{'N pairs':>10} {'Avg MSE':>12} {'Quality':>15}")
print("-" * 40)
for n, mse in zip(test_sizes, mses):
    if mse < 0.01:
        quality = "Excellent"
    elif mse < 0.1:
        quality = "Good"
    elif mse < 1.0:
        quality = "Fair"
    elif mse < 5.0:
        quality = "Poor"
    else:
        quality = "Terrible"
    print(f"{n:>10} {mse:>12.4f} {quality:>15}")

# Plot
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(test_sizes, mses, 'o-', color='#e53935', linewidth=2.5, markersize=8)
ax.axvline(x=d_k, color='gray', linestyle='--', alpha=0.7,
           label=f'State dimension d={d_k}')
ax.set_xlabel('Number of Stored Key-Value Pairs', fontsize=12)
ax.set_ylabel('Average Retrieval MSE', fontsize=12)
ax.set_title('Memory Degradation in Linear Attention', fontsize=14,
             fontweight='bold')
ax.legend(fontsize=11)
plt.tight_layout()
plt.show()

print(f"\n\u274c As more pairs are stored, retrieval quality degrades!")
print(f"   The state S accumulates noise from every write.")
print(f"   Past N \u2248 d = {d_k}, the memory is saturated.")
print(f"\nThis is the fundamental weakness of naive linear attention.")
print(f"We need a way to CORRECT the memory -- not just accumulate.")

In [ ]:
#@title 🎧 Code Walkthrough: Superposition Demo
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_37_superposition_demo.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
# Concrete example: the superposition problem
print("=" * 60)
print("  THE SUPERPOSITION PROBLEM")
print("=" * 60)

d = 2

# Start fresh
S = torch.zeros(d, d)

# Store "France" -> "Paris"
k_france = torch.tensor([1.0, 0.0])
v_paris = torch.tensor([0.8, 0.2])
S = S + torch.outer(k_france, v_paris)

# Store "Germany" -> "Berlin"
k_germany = torch.tensor([0.0, 1.0])
v_berlin = torch.tensor([0.3, 0.9])
S = S + torch.outer(k_germany, v_berlin)

print(f"After storing France->Paris and Germany->Berlin:")
retrieved = S.T @ k_france
print(f"  Query France: {retrieved.tolist()} (expected {v_paris.tolist()}) \u2705")

# Now UPDATE France -> Lyon (same key, new value)
v_lyon = torch.tensor([0.6, 0.7])
S = S + torch.outer(k_france, v_lyon)

retrieved = S.T @ k_france
print(f"\nAfter 'updating' France -> Lyon:")
print(f"  Query France: {retrieved.tolist()}")
print(f"  Expected (Lyon): {v_lyon.tolist()}")
print(f"  Got: Paris + Lyon = {(v_paris + v_lyon).tolist()}")
print(f"\n\u274c The old value 'Paris' was never erased!")
print(f"   The memory returns a superposition of old and new values.")
print(f"   This is the memory-as-a-whiteboard-with-no-eraser problem.")

print(f"\n\u2728 The solution: the DELTA RULE (Notebook 2)")
print(f"   Instead of writing v directly, write (v - current_prediction)")
print(f"   This turns the dumb accumulator into an error-correcting memory.")

---


In [ ]:
#@title 🎧 Narration: Reflection Takeaways
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_38_reflection_takeaways.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


## 9. \U0001f914 Reflection and Key Takeaways

### What We Learned

1. **Standard attention creates an $N \times N$ matrix** \u2014 the $QK^T$ product. This costs $O(N^2)$ in both compute and memory, making long sequences prohibitively expensive.

2. **Linear attention avoids the $N \times N$ matrix** by replacing softmax with a kernel and reordering the multiplication: instead of $(QK^T)V$ we compute $Q(K^T V)$. The cost drops to $O(N d_k d_v)$ \u2014 linear in $N$.

3. **The state matrix $S = K^T V$** is a $d_k \times d_v$ compressed summary of the sequence. It does not grow with $N$.

4. **Linear attention has an equivalent recurrent view**: $S_t = S_{t-1} + \tilde{k}_t v_t^T$, which makes inference $O(1)$ per token (constant cost, no KV cache).

5. **But the additive state update has a fatal flaw**: it can only accumulate, never correct or erase. Over time, the memory fills with noise and retrieval degrades.

### The Connection to Qwen3.5

Qwen3.5 uses **Gated DeltaNet** in 75% of its layers \u2014 a linear attention variant that solves the forgetting problem with two innovations:
- The **delta rule** (Notebook 2): Write only the error, not the raw value
- **Exponential gating** (Notebook 3): A learnable decay that enables selective forgetting

Together, these turn the simple state recurrence into a powerful, controllable memory.

### Reflection Questions

1. **Why can we reorder the multiplication in linear attention but not in softmax attention?** What property of softmax prevents this?

2. **The state matrix $S$ has $d_k \times d_v$ entries. What is the maximum number of independent key-value pairs it can store exactly?** (Hint: think about the rank of $S$.)

3. **Qwen3.5 uses a 3:1 ratio of linear-to-standard attention layers. Why not use 100% linear attention?** What would be lost?

---


In [ ]:
#@title 🎧 Transition: What Next
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_39_what_next.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


## 10. \U0001f680 What Comes Next

In **Notebook 2: DeltaNet \u2014 Error-Correcting Memory**, we will:

1. **See the forgetting problem in action** with a concrete key-value memory benchmark
2. **Implement the delta rule**: $S_t = S_{t-1} + k_t(v_t - S_{t-1}^T k_t)^T$ \u2014 write only the correction, not the raw value
3. **Benchmark naive vs delta rule** \u2014 showing orders-of-magnitude improvement in retrieval accuracy
4. **Visualize the state matrix** to understand WHY the delta rule works
5. **Understand the remaining limitation** \u2014 the delta rule corrects but cannot fully forget, motivating gating in Notebook 3

The linear attention framework gives us $O(1)$ inference per token. The delta rule will give us accurate memory. Together, they are two of the three ingredients in Qwen3.5\u2019s Gated DeltaNet layer.

In [ ]:
#@title 🎧 Wrap-Up: Closing
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_40_closing.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
# Final summary
print("""
\u2554\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2557
\u2551         FROM SOFTMAX TO LINEAR ATTENTION                    \u2551
\u2560\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2563
\u2551                                                               \u2551
\u2551  Standard Attention:                                          \u2551
\u2551    softmax(Q K^T / sqrt(d)) V     Cost: O(N^2 d)             \u2551
\u2551    Creates N x N matrix -- the bottleneck                     \u2551
\u2551                                                               \u2551
\u2551  Linear Attention:                                            \u2551
\u2551    Q_tilde @ (K_tilde^T @ V)      Cost: O(N d^2)             \u2551
\u2551    State S = K^T V -- fixed size d x d                        \u2551
\u2551                                                               \u2551
\u2551  Recurrent View:                                              \u2551
\u2551    S_t = S_{t-1} + k_t v_t^T      Cost: O(d^2) per token     \u2551
\u2551    output_t = q_t^T S_t            (constant -- no KV cache!) \u2551
\u2551                                                               \u2551
\u2551  Limitation:                                                   \u2551
\u2551    Purely additive -- no correction, no forgetting            \u2551
\u2551    Memory degrades with sequence length                       \u2551
\u2551    Solution: delta rule (Notebook 2)                          \u2551
\u2551                                                               \u2551
\u255a\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u255d
""")
print("\u2705 Notebook 1 complete! You now understand WHY we need linear attention")
print("   and HOW the multiplication reordering trick works.")
print("\U0001f4ca Next up: Notebook 2 -- DeltaNet: Error-Correcting Memory")